# Translate GoEmotions and SemEval labels

This notebook uses the raw GoEmotions data and SemEval training CSV from the original notebook. It looks up each active label by `Dataset` and `Base_Emotion` in the workbook and sets each of the nine target columns to `1` if any active label maps to it. For example, SemEval's `Love` sets both `Joy` and `Trust` to `1`.

Each input row produces one output row, including separate GoEmotions annotations that share a comment ID. Rows with no active source label have nine zeros. `Neutral` is set only when an active label maps to it.

## Inputs

Put `Emotion_Translations_Binary.xlsx` and `SemEval-Train.csv` in the notebook's working directory, or edit their paths below. In Colab, upload both files to the session. The notebook loads a local `go_emotions_raw.parquet` if present, or fetches the [raw GoEmotions file](https://huggingface.co/datasets/google-research-datasets/go_emotions/tree/main/raw) from Hugging Face.

The Python kernel needs `pandas` (version 2 or later), `openpyxl`, and `pyarrow`. If needed, install them in a notebook cell with `%pip install "pandas>=2" openpyxl pyarrow`.

In [ ]:
from pathlib import Path

import pandas as pd

MAPPING_PATH = Path("Emotion_Translations_Binary.xlsx")
SEMEVAL_PATH = Path("SemEval-Train.csv")
GOEMOTIONS_LOCAL_PATH = Path("go_emotions_raw.parquet")
GOEMOTIONS_URL = (
    "https://huggingface.co/datasets/google-research-datasets/go_emotions/"
    "resolve/main/raw/train-00000-of-00001.parquet"
)
OUTPUT_PATH = Path("GoEmotions_SemEval_Binary.csv")

TARGET_EMOTIONS = [
    "Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust",
    "Anger", "Anticipation", "Neutral",
]

# Base label columns in the raw GoEmotions file and SemEval training CSV.
# Lowercase names match workbook labels and dataset headers case-insensitively.
BASE_EMOTIONS = {
    "GoEmotions": [
        "admiration", "amusement", "anger", "annoyance", "approval",
        "caring", "confusion", "curiosity", "desire", "disappointment",
        "disapproval", "disgust", "embarrassment", "excitement", "fear",
        "gratitude", "grief", "joy", "love", "nervousness", "optimism",
        "pride", "realization", "relief", "remorse", "sadness", "surprise",
        "neutral",
    ],
    "SemEval": [
        "anger", "anticipation", "disgust", "fear", "joy", "love",
        "optimism", "pessimism", "sadness", "surprise", "trust",
    ],
}

## Read and validate the translation table

The workbook's `Mappings` sheet supplies the target flags. The notebook checks that each expected base label has exactly one mapping row for its dataset and that the flags are binary.

In [ ]:
def binary_flags(values, source_name):
    numeric = values.apply(pd.to_numeric, errors="coerce")
    invalid_columns = numeric.columns[~numeric.isin([0, 1]).all()]
    if len(invalid_columns):
        raise ValueError(
            f"{source_name} must contain only 0 or 1 in its label columns. "
            f"Invalid columns: {', '.join(invalid_columns)}"
        )
    return numeric.astype("uint8")


mapping = pd.read_excel(
    MAPPING_PATH, sheet_name="Mappings", engine="openpyxl", keep_default_na=False
)
missing_columns = {"Dataset", "Base_Emotion", *TARGET_EMOTIONS} - set(mapping)
if missing_columns:
    raise ValueError(f"Missing workbook columns: {sorted(missing_columns)}")

for key in ["Dataset", "Base_Emotion"]:
    mapping[key] = mapping[key].astype("string").str.strip().str.casefold()

translations = {}
for dataset, base_labels in BASE_EMOTIONS.items():
    lookup = mapping.loc[mapping["Dataset"].eq(dataset.casefold())]
    lookup = lookup.set_index("Base_Emotion")[TARGET_EMOTIONS]
    if not lookup.index.is_unique:
        duplicates = lookup.index[lookup.index.duplicated()].unique().tolist()
        raise ValueError(f"{dataset} has duplicate workbook mappings: {duplicates}")

    missing = set(base_labels) - set(lookup.index)
    unexpected = set(lookup.index) - set(base_labels)
    if missing or unexpected:
        raise ValueError(
            f"{dataset} workbook mappings do not match its base labels. "
            f"Missing: {sorted(missing)}. Unexpected: {sorted(unexpected)}."
        )
    translations[dataset] = binary_flags(
        lookup.loc[base_labels], f"{dataset} workbook mappings"
    )

## Translate and concatenate

The combined table starts with the dataset name, text, and nine translated labels. Other source columns follow, including GoEmotions comment and rater IDs.

In [ ]:
def translate_labels(frame, dataset, lookup):
    headers = {name.strip().casefold(): name for name in frame.columns}
    if len(headers) != len(frame.columns):
        raise ValueError(f"{dataset} has duplicate column names after normalization.")

    required = {"text", *lookup.index}
    missing = required - headers.keys()
    if missing:
        raise ValueError(f"{dataset} is missing input columns: {sorted(missing)}")

    source_columns = [headers[label] for label in lookup.index]
    source_flags = binary_flags(frame[source_columns], f"{dataset} input")
    source_flags.columns = lookup.index

    # Each product counts active source labels that map to a target.
    # Any positive count becomes 1, including overlapping mappings.
    targets = (source_flags @ lookup).gt(0).astype("uint8")

    metadata = frame.drop(columns=source_columns)
    metadata = metadata.rename(columns={headers["text"]: "text"})
    metadata.insert(0, "Dataset", dataset)
    collisions = set(metadata) & set(TARGET_EMOTIONS)
    if collisions:
        raise ValueError(f"{dataset} metadata conflicts with target columns: {sorted(collisions)}")

    return pd.concat(
        [metadata[["Dataset", "text"]], targets, metadata.drop(columns=["Dataset", "text"])],
        axis=1,
    )


go_source = GOEMOTIONS_LOCAL_PATH if GOEMOTIONS_LOCAL_PATH.is_file() else GOEMOTIONS_URL
df_go = pd.read_parquet(go_source, engine="pyarrow", dtype_backend="numpy_nullable")
df_semeval = pd.read_csv(
    SEMEVAL_PATH, encoding="utf-8-sig", dtype={"text": str}, keep_default_na=False
)

df_go_binary = translate_labels(df_go, "GoEmotions", translations["GoEmotions"])
df_semeval_binary = translate_labels(df_semeval, "SemEval", translations["SemEval"])
df_combined = pd.concat([df_go_binary, df_semeval_binary], ignore_index=True)

## Review counts and save the combined CSV

In [ ]:
grouped = df_combined.groupby("Dataset", sort=False)
summary = grouped[TARGET_EMOTIONS].sum()
summary.insert(0, "Rows", grouped.size())
no_active_targets = df_combined[TARGET_EMOTIONS].sum(axis=1).eq(0)
summary["All-zero rows"] = no_active_targets.groupby(df_combined["Dataset"]).sum()
print(summary.to_string())

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_combined.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df_combined):,} rows to {OUTPUT_PATH.resolve()}")